## Category segmentation with o3 modell

Prompting the LLM modell

You must categorize the product list below and return the result in CSV format, indicating which stock code belongs to which category. Create main categories and sub-categories where necessary, and assign no more than two sub-categories to any main category.

In [0]:
import pandas as pd

In [0]:
xlsx_path = "/Volumes/main_catalog/online_retail/raw/hierarchie_tbl.xlsx"

# Read the Excel file into a pandas DataFrame
hier_pd = pd.read_excel(xlsx_path, engine="openpyxl")
hier_pd['StockCode'] = hier_pd['StockCode'].astype(str)

hier_df = spark.createDataFrame(hier_pd)
(hier_df.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable("main_catalog.online_retail.article_hierarchy"))

In [0]:
%sql 

SELECT *
FROM main_catalog.online_retail.article_hierarchy h LIMIT 100;

In [0]:
%sql

SELECT 
 h.MainCategory_hun AS category, 
 SUM(trx.Quantity) AS total_sold,
 SUM(trx.Price*trx.Quantity) AS total_revenue,
 COUNT(DISTINCT trx.Invoice) AS total_trx_nr,
 COUNT(DISTINCT trx.StockCode) as total_articles
FROM main_catalog.online_retail.sales_silver trx
JOIN main_catalog.online_retail.article_hierarchy h ON trx.StockCode=h.StockCode
GROUP BY h.MainCategory_hun
ORDER BY total_sold DESC 